In [ ]:
def mede_stress(sigma,tau):
    """
    Compute the stress criterion for a FCDH weak layer as given in Mede et. al. (2018).
    Instead of a linear branch, a full quadratic interaction criterion is used.
    Arguments
    ---------
    sigma: float or ndarray
        Compressive stress (kPa)
    tau: float or ndarray
        Shear stress (kPa)    

    Returns
    -------
    f: float or ndarray
        Stress criterion
    """
    tauPeak = 0.61
    sigPeak = 0.17
    sigMax = 1.45
    return ((sigma-sigPeak)/(sigMax))**2 + (tau/tauPeak)**2


In [ ]:
def energy_crit(g1,g2):
    """
    Evaluate a mixed-mode interaction criterion for the energy release rates
    gIc = 0.56+-0.02 J/m²
    gIIc = 0.79+-0.04 J/m²
    Arguments
    ---------
    g1: float
        Mode I ERR (J/m^2)
    g2: float
        Mode II ERR (J/m^2)
    Returns:
    -------
    g: float
        Evluated energy crieterion
    """
    gIc = 0.56
    gIIc = 0.79
    g = (g1/gIc)**5.0 + (g2/gIIc)**2.2
    return g

In [ ]:
def ffm_evaluate(sys, m, phi, totalLength=1e4,num=4):
    """
    Evaluate stress and energy criterion. 
    Arguments:
        sys: Layered class object
            An object of Layered
        m:  float
            Applied skier mass (kg)
        phi: float
            Inclination (°)
        totalLength: float
            Total length of the block (mm)
        
    Returns
    -------
    ac: float
        IncitialCrack length for given mass
    G/Gc: float
        Evaluated energy criterion 
    """
    # Identify overloaded area
    segmentsInitial = sys.calc_segments(li = [totalLength/2., totalLength/2.],ki = [True,  True],
                              k0 = [True, True],
                              mi = [m], qi = [True, True])
    C0 = sys.assemble_and_solve(phi=phi,**segmentsInitial['nocrack'])
    xwl,z0,xsl = sys.rasterize_solution(C0,phi=phi,num=int(totalLength*num),**segmentsInitial['nocrack'])
    # Evaluate stress criterion
    stress_crit = mede_stress(-sys.sig(z0,unit='kPa'),sys.tau(z0,unit='kPa'))
    
    #stress_crit_inter = interpolate.interp1d(xsl,stress_crit-1., kind= 'cubic')
    overload=(stress_crit-1)>1e-3
    
    if not np.any(overload):
        print('Problem 1')
        raise FFMError('Increase Load', 0)
        
    elif np.all(overload):
        print('Problem 2')
        raise FFMError(' At inclination pi = '+ str(phi)+ ' the whole weak layer is overloaded.',1)
    startOverload = xsl[np.where(overload)[0][0]]
    endOverload= xsl[np.where(overload)[0][-1]]
    
    # fp = mede_stress(-sys.sig(z0[:,np.where(overload)[0][0]],unit='kPa'),sys.tau(z0[:,np.where(overload)[0][0]],unit='kPa'))
    # fm = mede_stress(-sys.sig(z0[:,np.where(overload)[0][0]-1],unit='kPa'),sys.tau(z0[:,np.where(overload)[0][0]-1],unit='kPa'))
    # xm = xsl[np.where(overload)[0][0]-1]
    # xp = xsl[np.where(overload)[0][0]]

    # startOverload=xm+((1-fm)/(fp-fm)*(xp-xm))
    

    # fpe = mede_stress(-sys.sig(z0[:,np.where(overload)[0][-1]],unit='kPa'),sys.tau(z0[:,np.where(overload)[0][-1]],unit='kPa'))
    # fme = mede_stress(-sys.sig(z0[:,np.where(overload)[0][-1]+1],unit='kPa'),sys.tau(z0[:,np.where(overload)[0][-1]+1],unit='kPa'))
    # xme = xsl[np.where(overload)[0][-1]]
    # xpe = xsl[np.where(overload)[0][-1]+1]
    # endOverload=xme+((1-fme)/(fpe-fme)*(xpe-xme))
    
    # startOverload=optimize.brentq(stress_crit_inter,a=0,b=totalLength/2.)
    # endOverload=optimize.brentq(stress_crit_inter,a=totalLength/2.,b=totalLength)
    
    ac = endOverload-startOverload
    # Compute incremental energy release rate
    segmentsOverload=sys.calc_segments(li = [startOverload,totalLength/2.-startOverload,endOverload-totalLength/2.,totalLength-endOverload],
                                   mi = [0,m,0],
                                   ki = [True, False, False, True],
                                   k0 = [True, True, True, True],
                                   qi = [True, True, True, True])
    C1 = sys.assemble_and_solve(phi=phi,**segmentsOverload['nocrack'])
    C2 = sys.assemble_and_solve(phi=phi,**segmentsOverload['crack'])
    # xwl1,z1,xsl1 = sys.rasterize_solution(C1,phi=phi,num=2000,**segmentsOverload['nocrack'])
    # xwl2,z2,xsl2 = sys.rasterize_solution(C2,phi=phi,num=2000,**segmentsOverload['crack'])
    
    ginc = sys.ginc(C1,C2,phi=phi, **segmentsOverload['both'])*1000
    g = energy_crit(ginc[1],ginc[2])
  
    return g, ac, ginc, startOverload, endOverload


In [ ]:
def err_evaluate(sys, m, phi, sp, ep, totalLength=1e4):
    """
    Evaluate stress and energy criterion. 
    Arguments:
        sys: Layered class object
            An object of Layered
        m:  float
            Applied skier mass (kg)
        phi: float
            Inclination (°)
        totalLength: float
            Total length of the block (mm)
        
    Returns
    -------
    ac: float
        IncitialCrack length for given mass
    G/Gc: float
        Evaluated energy criterion 
    """
    # Compute incremental energy release rate
    segmentsOverload=sys.calc_segments(li = [sp,totalLength/2.-sp,ep-totalLength/2.,totalLength-ep],
                                   mi = [0,m,0],
                                   ki = [True, False, False, True],
                                   k0 = [True, True, True, True],
                                   qi = [True, True, True, True])
    C1 = sys.assemble_and_solve(phi=phi,**segmentsOverload['nocrack'])
    C2 = sys.assemble_and_solve(phi=phi,**segmentsOverload['crack'])
    # xwl1,z1,xsl1 = sys.rasterize_solution(C1,phi=phi,num=2000,**segmentsOverload['nocrack'])
    # xwl2,z2,xsl2 = sys.rasterize_solution(C2,phi=phi,num=2000,**segmentsOverload['crack'])
    
    ginc = sys.ginc(C1,C2,phi=phi, **segmentsOverload['both'])*1000
    g = energy_crit(ginc[1],ginc[2])
  
    return g, ginc


In [ ]:
def optimizer(ffm,phi):
    massInitial = 150.
    
    spMin = 1e4
    epMin = 1e4
    spMax = 0
    epMax = 0
    g=0
    gInitial=0
    while not gInitial>1:
        gInitial,ac,ginc,sP,eP = ffm_evaluate(ffm,massInitial,phi)
        if gInitial<1:
            massInitial =2*massInitial


    massRange =[0, massInitial]
    gRange = [0,gInitial]
    spRange = [0.,1e4]
    epRange = [0, 1e4]
    num=5
    crackOptimizer = False
    counter =0
    while abs(g-1)>=1e-3:
        counter += 1
        mass = np.mean(massRange)
        try:
            g,ac,ginc,sP,eP = ffm_evaluate(ffm,mass,phi,num=num)
            
        except FFMError as e:
            if e.code ==0:
                massRange[0] = mass
                continue
            elif e.code ==1:
                massRange[1]=mass
                continue                
            
        #print(counter, mass, ac, g)
        if g>1:
            massRange[1] = mass
            gRange[1] = g
            spRange[0] = sP
            epRange[1] = eP
        elif g<1:
            massRange[0] = mass
            gRange[0] = g
            spRange[1] = sP
            epRange[0] = eP

        
        if np.diff(massRange)[0]<1e-6:
            if np.diff(gRange)[0]<1e-4:
                break
            elif np.diff(gRange)[0]<1:
                if np.diff(spRange)[0]<1e-2 and np.diff(epRange)[0]<1e-2:
                    mass = massRange[0]
                    break
            elif np.diff(gRange)[0]>1:
                crackOptimizer=True
                break

    if crackOptimizer:
        while abs(g-1)>1e-2:
            counter +=1
            sP = np.mean(spRange)
            eP = np.mean(epRange)

            g,ginc = err_evaluate(ffm,massRange[1],phi,sP,eP)
            
            if g>1:
                spRange[0] = sP
                epRange[1] = eP
                gRange[1]=g
            elif g<1:
                spRange[1] = sP
                epRange[0] = eP
                gRange[0] = g
            
            if abs(np.diff(spRange)[0])<1e-6 and abs(np.diff(epRange)[0])<1e-6:
                #print(gRange,np.array(gRange)-1)
                ac = epRange[np.argmin(np.abs(np.array(gRange)-1))]-spRange[np.argmax(np.abs(np.array(gRange)-1))]
                break
    ac = eP-sP
    return mass, ac              
    
            
            

            
            

